# Agentic DocQA: GPU evaluation
Runs the real document pipeline, with no Docker or web server. This notebook has not been executed on a Colab GPU by the authoring agent.

Use **Runtime > Change runtime type > T4 GPU**. The first run downloads packages and models. Each completed question is saved directly to Drive and later runs skip compatible completed questions. The default source is the ZIP in your shared **My Drive/Agentic DocQA** folder. No GitHub token is needed for that option.

Results go to **My Drive/agentic-docqa-eval**. This evaluates 20 synthetic questions; it is a diagnostic, not a production benchmark. A same-model judge can be biased. No evaluation scores are included in this notebook.


## 1. Check the GPU
Stop early if no NVIDIA GPU is assigned. Colab GPU availability and runtime length are not guaranteed.


In [ ]:
import subprocess, sys
check = subprocess.run(["bash", "-lc", "nvidia-smi"], text=True, capture_output=True)
print(check.stdout or check.stderr)
print("Python:", sys.version)
if check.returncode:
    raise RuntimeError("Choose Runtime > Change runtime type > T4 GPU, then rerun this notebook.")


## 2. Load the private project
**Drive ZIP** loads the project already saved in your Drive. **Upload ZIP** lets you select a downloaded project ZIP. Both avoid GitHub authentication. For **GitHub**, optionally add a repository-scoped token as the Colab Secret `GITHUB_TOKEN` and allow this notebook to read it. Tokens are never put in URLs, shell commands, outputs, or saved files.

Use the updated ZIP containing `eval/run_eval.py`; the older project-only ZIP cannot run the evaluation.


In [ ]:
import hashlib, io, os, shutil, stat, urllib.request, zipfile
from pathlib import Path, PurePosixPath
SOURCE = "Drive ZIP" #@param ["Drive ZIP", "Upload ZIP", "GitHub"]
DRIVE_ZIP = "/content/drive/MyDrive/Agentic DocQA/agentic-docqa.zip" #@param {type:"string"}
if SOURCE == "Drive ZIP":
    from google.colab import drive
    drive.mount("/content/drive")
    archive_bytes = Path(DRIVE_ZIP).read_bytes()
elif SOURCE == "Upload ZIP":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select exactly one updated project ZIP.")
    archive_bytes = next(iter(uploaded.values()))
else:
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = "Bearer " + token
    request = urllib.request.Request("https://api.github.com/repos/arpitpaliwal007/agentic-docqa/zipball/main", headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            archive_bytes = response.read()
    except Exception:
        raise RuntimeError("Could not load GitHub source. For this private repository, use Drive ZIP or grant the GITHUB_TOKEN secret repository read access.") from None
    finally:
        headers.clear()
        request = None
        token = None
REPO = Path("/content/agentic-docqa-eval-source") / hashlib.sha256(archive_bytes).hexdigest()[:16]
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    entries = []
    roots = set()
    for entry in archive.infolist():
        path = PurePosixPath(entry.filename)
        if path.is_absolute() or ".." in path.parts or not path.parts or stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError("Unsafe ZIP entry")
        roots.add(path.parts[0])
        entries.append((entry, path.parts[1:]))
    if len(roots) != 1:
        raise ValueError("Expected one project folder inside the ZIP")
    for entry, parts in entries:
        if not parts or ".git" in parts:
            continue
        target = REPO.joinpath(*parts)
        if target.is_symlink() or any(parent.is_symlink() for parent in target.parents):
            raise ValueError("Unsafe extraction target")
        if entry.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(entry))
if not (REPO / "eval/run_eval.py").is_file():
    raise RuntimeError("This ZIP predates the evaluation harness. Upload or save the updated project ZIP and rerun.")
os.chdir(REPO)
print("Project loaded:", REPO)


## 3. Install dependencies and validate fixtures
No extra evaluation packages are required. Install in a subprocess; evaluation also runs in a fresh Python subprocess to avoid stale imports after package changes. GPU-dependent checks remain in the smoke test.

Google Drive also stores package/model download caches so they survive a new runtime.


In [ ]:
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
MODEL_CACHE_ROOT = Path("/content/drive/MyDrive/Agentic DocQA")
RESULTS_DIR = Path("/content/drive/MyDrive/agentic-docqa-eval")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.update({"OLLAMA_MODELS":str(MODEL_CACHE_ROOT / "models/ollama"), "HF_HOME":str(MODEL_CACHE_ROOT / "cache/huggingface"), "FASTEMBED_CACHE_PATH":str(MODEL_CACHE_ROOT / "cache/fastembed"), "SENTENCE_TRANSFORMERS_HOME":str(MODEL_CACHE_ROOT / "cache/sentence-transformers"), "PIP_CACHE_DIR":str(MODEL_CACHE_ROOT / "cache/pip")})
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "eval/run_eval.py", "--validate-only"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## 4. Install and start Ollama
Use the official installer, start the server in the background if needed, and poll readiness for up to 60 seconds. Ollama logs stay in `My Drive/agentic-docqa-eval/ollama.log`. Model context is set to 8192 tokens and parallel requests to one to limit GPU memory use.


In [ ]:
import json, time
OLLAMA_URL = "http://127.0.0.1:11434"
def ollama_json(path, payload=None, timeout=10):
    data = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(OLLAMA_URL + path, data=data, headers={"Content-Type":"application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)
def ollama_ready():
    try:
        ollama_json("/api/tags", timeout=2)
        return True
    except Exception:
        return False
if not shutil.which("ollama"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
    installer = Path("/content/install-ollama-eval.sh")
    urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
    subprocess.run(["sh", str(installer)], check=True)
os.environ.update({"OLLAMA_HOST":"127.0.0.1:11434", "OLLAMA_CONTEXT_LENGTH":"8192", "OLLAMA_NUM_PARALLEL":"1", "OLLAMA_MAX_LOADED_MODELS":"2"})
if not ollama_ready():
    with open(str(RESULTS_DIR / "ollama.log"), "a") as log:
        ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy(), start_new_session=True)
    for _ in range(60):
        if ollama_ready():
            break
        if ollama_process.poll() is not None:
            raise RuntimeError(Path(str(RESULTS_DIR / "ollama.log")).read_text()[-3000:])
        time.sleep(1)
    else:
        raise RuntimeError("Ollama did not respond within 60 seconds. Check the log.")
print("Ollama ready.")


## 5. Choose and pull models
The default is `qwen2.5:7b`; choose `qwen2.5:3b` for a lighter run. Embeddings always use `nomic-embed-text`. Leave the judge field empty to use the answer model as judge; the report explicitly labels that self-judging limitation. An optional different judge also has to fit the runtime's resources.


In [ ]:
MODEL = "qwen2.5:7b" #@param ["qwen2.5:7b", "qwen2.5:3b"]
JUDGE_MODEL = "" #@param {type:"string"}
os.environ.update({"EVAL_LLM_MODEL":MODEL, "LLM_MODEL":MODEL, "EVAL_JUDGE_MODEL":JUDGE_MODEL or MODEL, "LLM_BASE_URL":OLLAMA_URL + "/v1", "LLM_API_KEY":"ollama", "OLLAMA_HOST":OLLAMA_URL})
for model in dict.fromkeys([MODEL, "nomic-embed-text", JUDGE_MODEL or MODEL]):
    subprocess.run(["ollama", "pull", model], check=True)
ollama_json("/api/generate", {"model":MODEL,"prompt":"Reply with OK.","stream":False,"keep_alive":"30m","options":{"num_predict":8,"num_ctx":8192}}, timeout=600)
loaded = ollama_json("/api/ps")["models"]
if not any(item.get("size_vram", 0) > 0 and item.get("name") == MODEL for item in loaded):
    raise RuntimeError("The answer model was not loaded onto the GPU. Check the Ollama log/driver before evaluating.")
print("Answer model GPU use verified.")


## 6. Mount Drive and choose durable result storage
Completed questions are saved directly here, including raw evidence and judge outputs. The live embedded Qdrant index stays on local runtime disk, isolated from your normal DocQA library. A fresh runtime rebuilds the tiny corpus, then skips compatible completed questions. Package versions and GPU details are part of run identity.


In [ ]:
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
RESULTS_DIR = Path("/content/drive/MyDrive/agentic-docqa-eval")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Durable results:", RESULTS_DIR)


## 7. Smoke test: two questions
This executes indexing, all three retrieval variants, the real graph, and judging. Smoke results use a separate directory so they cannot be mistaken for the full evaluation. A setup, tool, or judge failure makes this cell fail.


In [ ]:
subprocess.run([sys.executable,"eval/run_eval.py","--model",MODEL,"--judge-model",JUDGE_MODEL or MODEL,"--limit","2","--out",str(RESULTS_DIR / "smoke")], check=True)


## 8. Full evaluation (resumable)
Rerun this cell after reconnecting and repeating setup. Each completed question is checkpointed; interrupted or errored questions are retried. Changing the corpus, code, model, dependencies, GPU, or grading settings creates a different run fingerprint. No normal application data is cleared.


In [ ]:
subprocess.run([sys.executable,"eval/run_eval.py","--model",MODEL,"--judge-model",JUDGE_MODEL or MODEL,"--out",str(RESULTS_DIR)], check=True)


## 9. Display the report
The Markdown report includes observed scores, per-question results, retrieval comparison, CRAG counts and up to three observed failures. Full evidence, configuration and raw judge responses are preserved in the matching JSON file. Results are never prefilled.


In [ ]:
from IPython.display import Markdown, display
reports = sorted(RESULTS_DIR.glob("*.md"), key=lambda path:path.stat().st_mtime)
if not reports:
    raise RuntimeError("No full-run report exists yet. Run the smoke/full evaluation cells first.")
report = reports[-1]
print("Report:", report)
display(Markdown(report.read_text()))
